# Lab 12: structured output

A JSON schema on the request, the first-request grammar cost, what the grammar does to the step, and a failure mode you should see once: the grammar lets the model pad with whitespace.

In [ ]:
# Cell 1: the GPU this notebook got. On Kaggle: Settings → Accelerator → GPU T4 x2, then Run all.
import subprocess, sys
out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"], capture_output=True, text=True)
if out.returncode != 0 or not out.stdout.strip():
    sys.exit("No GPU. Open Settings → Accelerator and choose GPU T4 x2, then run this cell again.")
print(out.stdout.strip())
if "T4" in out.stdout:
    print("Path: Kaggle T4. Lab 10 is read-only on this GPU (no FP8 KV cache); the FP8 checkpoint in Lab 08 runs with unpacked weights.")

In [ ]:
# Cell 2: install the pinned engine. About nine minutes on Kaggle. Kaggle's preinstalled torchaudio is built
# for an older CUDA and breaks the import, so it is removed.
%pip install -q "vllm==0.32.0" 2>&1 | tail -2
%pip uninstall -y -q torchaudio
import vllm, torch
print("vllm", vllm.__version__, "· torch", torch.__version__, "· compute capability", torch.cuda.get_device_capability())

In [ ]:
# Cell 3: helpers used by every lab: start the server in the background, wait for it, talk to it.
import json, re, subprocess, time, urllib.request
MODEL = "Qwen/Qwen3-4B-Instruct-2507"
URL = "http://127.0.0.1:8000"

def serve(extra_args=(), log_path="/kaggle/working/serve.log", timeout_s=900):
    args = ["vllm", "serve", MODEL, "--port", "8000", "--dtype", "half", *extra_args]   # the T4 has no bfloat16
    log = open(log_path, "w")
    t0 = time.time()
    proc = subprocess.Popen(args, stdout=log, stderr=subprocess.STDOUT)
    while time.time() - t0 < timeout_s:
        if proc.poll() is not None:
            raise RuntimeError("vllm serve exited; see " + log_path)
        try:
            urllib.request.urlopen(URL + "/health", timeout=2); return proc, round(time.time() - t0, 1)
        except Exception:
            time.sleep(0.5)
    raise RuntimeError("server not ready in time")

def stop(proc):
    proc.terminate(); proc.wait()

def log_lines(path, patterns):
    text = open(path).read(); out = {}
    for name, pat in patterns.items():
        m = re.search(pat, text)
        if m:
            line = re.sub(r"^\(EngineCore[^)]*\) ", "", m.group(0)); line = re.sub(r"^INFO \d\d-\d\d \d\d:\d\d:\d\d \[[^\]]+\] ", "", line); out[name] = line.strip()
    return out

STARTUP = {"dtype": r"INFO[^\n]*dtype=torch\.[a-z0-9]+", "backend": r"Using [A-Z_]+ attention backend[^\n]*", "weights": r"Loading weights took [^\n]*",
           "model": r"Model loading took [^\n]*", "kv_memory": r"Available KV cache memory: [^\n]*", "kv_size": r"GPU KV cache size: [^\n]*", "graphs": r"Graph capturing finished[^\n]*"}

def chat(messages, **params):
    body = json.dumps({"model": MODEL, "messages": messages, **params}).encode()
    req = urllib.request.Request(URL + "/v1/chat/completions", data=body, headers={"Content-Type": "application/json"})
    return json.loads(urllib.request.urlopen(req, timeout=300).read())
print("helpers ready")

In [ ]:
def stream_chat(messages, **params):
    body = json.dumps({"model": MODEL, "messages": messages, "stream": True, **params}).encode()
    req = urllib.request.Request(URL + "/v1/chat/completions", data=body, headers={"Content-Type": "application/json"})
    t0 = time.time(); first = None; last = None; gaps = []; text = []
    with urllib.request.urlopen(req, timeout=600) as resp:
        for raw in resp:
            line = raw.decode().strip()
            if not line.startswith("data:") or line == "data: [DONE]": continue
            delta = json.loads(line[5:])["choices"][0].get("delta", {}).get("content")
            if not delta: continue
            now = time.time()
            if first is None: first = now - t0
            else: gaps.append(now - last)
            last = now; text.append(delta)
    return first, gaps, "".join(text)

def metrics(names):
    out = {}
    for line in urllib.request.urlopen(URL + "/metrics", timeout=5).read().decode().splitlines():
        if line.startswith("#"): continue
        for n in names:
            if line.startswith(n + "{") or line.startswith(n + " "):
                try: out[n] = float(line.rsplit(" ", 1)[1])
                except ValueError: pass
    return out
print("streaming and metrics helpers ready")

In [ ]:
# Cell 5: two schemas and the request.
SIMPLE = {"type": "object", "properties": {"name": {"type": "string"}, "age": {"type": "integer"}, "city": {"type": "string"}}, "required": ["name", "age", "city"], "additionalProperties": False}
NESTED = {"type": "object", "properties": {"customer": {"type": "object", "properties": {"name": {"type": "string"}, "email": {"type": "string"}, "address": {"type": "object", "properties": {"street": {"type": "string"}, "city": {"type": "string"}, "postcode": {"type": "string"}}, "required": ["street", "city", "postcode"]}}, "required": ["name", "email", "address"]},
  "items": {"type": "array", "items": {"type": "object", "properties": {"sku": {"type": "string"}, "quantity": {"type": "integer"}, "unit_price": {"type": "number"}}, "required": ["sku", "quantity", "unit_price"]}, "minItems": 2}, "total": {"type": "number"}, "paid": {"type": "boolean"}}, "required": ["customer", "items", "total", "paid"], "additionalProperties": False}
PROMPT = "Invent an order for a small online shop with a customer in Pune and two items, as JSON."
def ask(schema=None, name="s"):
    params = {"max_tokens": 200, "temperature": 0}
    if schema: params["response_format"] = {"type": "json_schema", "json_schema": {"name": name, "schema": schema}}
    first, gaps, text = stream_chat([{"role": "user", "content": PROMPT}], **params); ok = "n/a"
    if schema:
        try: json.loads(text); ok = "valid JSON"
        except Exception: ok = "INVALID: ends with " + repr(text[-30:])
    return first, gaps, text, ok
def rounds(title, schema, name):
    print(title)
    for i in range(2):
        first, gaps, text, ok = ask(schema, name); print(f"  request {i + 1}: first token {1000 * first:.0f} ms, time per token {1000 * sum(gaps) / len(gaps):.1f} ms, {len(gaps) + 1} tokens, {ok}")
    print("  " + text.replace("\n", " ")[:300])

In [ ]:
# Cell 6: no schema, then the two schemas, twice each.
proc, ready_s = serve(["--max-model-len", "4096"]); print(f"ready after {ready_s} s")
first, gaps, text, _ = ask(); print(f"no schema: first token {1000 * first:.0f} ms, time per token {1000 * sum(gaps) / len(gaps):.1f} ms, {len(gaps) + 1} tokens\n  " + text.replace("\n", " ")[:200])
rounds("simple schema", SIMPLE, "simple"); rounds("nested schema", NESTED, "nested")
stop(proc)

In [ ]:
# Cell 7: the same with any-whitespace disabled in the grammar.
proc, ready_s = serve(["--max-model-len", "4096", "--structured-outputs-config", json.dumps({"backend": "xgrammar", "disable_any_whitespace": True})]); print(f"ready after {ready_s} s")
rounds("simple schema", SIMPLE, "simple"); rounds("nested schema", NESTED, "nested")
stop(proc)